In [211]:
%pip install -q git+https://github.com/tommasocarraro/LTNtorch.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [212]:
#!/usr/bin/env python
# coding: utf-8

# In[32]:
import torch
import torch.nn as nn
import ltn
import numpy as np
import matplotlib.pyplot as plt

In [213]:
# In[33]:
# Configure o LTNtorch para usar o dispositivo apropriado.
ltn.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {ltn.device}")

Using device: cpu


In [214]:
# In[34]:
# 1. DEFINIR CONECTIVOS LÓGICOS E QUANTIFICADORES (REQUERIDOS NO LTNTORCH)
Not = ltn.Connective(ltn.fuzzy_ops.NotStandard())
And = ltn.Connective(ltn.fuzzy_ops.AndProd())
Or = ltn.Connective(ltn.fuzzy_ops.OrProbSum())
Implies = ltn.Connective(ltn.fuzzy_ops.ImpliesReichenbach())
Forall = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
Exists = ltn.Quantifier(ltn.fuzzy_ops.AggregPMean(p=2), quantifier="e")
sat_agg = ltn.fuzzy_ops.SatAgg()

In [215]:
# In[35]:
# 2. CRIAR OBJETOS LTN (Constantes e Variáveis)
red_ball = ltn.Constant(torch.tensor([1.0, 0.0, 0.0]), trainable=False)
blue_cube = ltn.Constant(torch.tensor([0.0, 0.0, 1.0]), trainable=False)

balls = ltn.Variable("balls", torch.tensor([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.5, 0.0, 0.5],
]))
cubes = ltn.Variable("cubes", torch.tensor([
    [0.0, 0.0, 1.0],
    [1.0, 1.0, 0.0],
    [0.5, 0.5, 0.5],
]))

In [216]:
# In[36]:
# 3. DEFINIR PREDICADOS (Propriedades/relações lógicas)
class ColorPredicate(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(3, 10),
            nn.ReLU(),
            nn.Linear(10, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        if hasattr(x, 'value'):
            x = x.value
        return self.network(x).squeeze()

In [217]:
is_red = ltn.Predicate(ColorPredicate())
is_blue = ltn.Predicate(ColorPredicate())
is_round = ltn.Predicate(ColorPredicate())

In [218]:
# 4. DEFINIR AXIOMAS LÓGICOS (Base de conhecimento)
def axioms():
    facts = [
        is_red(red_ball),
        is_blue(blue_cube),
        is_round(red_ball),
        Not(is_round(blue_cube))
    ]
    rules = [
        Forall(balls, is_round(balls)),
        Forall(ltn.Variable("x", torch.randn(10, 3, device=ltn.device)),
            Implies(is_red(ltn.Variable("x", torch.randn(10, 3, device=ltn.device))),
            is_round(ltn.Variable("x", torch.randn(10, 3, device=ltn.device)))))
    ]
    all_axioms = facts + rules
    return sat_agg(*all_axioms)

In [219]:
# 5. VERIFICAÇÃO DE SATISFATÓRIA (Raciocínio)
def check_satisfiability():
    sat_level = axioms()
    print(f"Knowledge base satisfaction level: {sat_level.item():.4f}")
    return sat_level

In [220]:
# 6. TREINAMENTO DOS PREDICADOS (Aprendendo com os dados)
def train_predicates(epochs=100):
    parameters = list(is_red.model.parameters()) + \
                 list(is_blue.model.parameters()) + \
                 list(is_round.model.parameters())
    optimizer = torch.optim.Adam(parameters, lr=0.01)
    print("\n=== TRAINING PREDICATES ===")
    for epoch in range(epochs):
        optimizer.zero_grad()
        sat_loss = 1 - axioms()
        sat_loss.backward()
        optimizer.step()
        if epoch % 20 == 0:
            print(f"Epoch {epoch}: Satisfaction = {1-sat_loss.item():.4f}")

In [221]:
# In[43]:
# 7. QUERYING THE KNOWLEDGE BASE (Inference)
def query_knowledge_base():
    print("\n=== QUERYING KNOWLEDGE BASE ===")
    # Query 1: Is the red ball red?
    query1 = is_red(red_ball)
    print(f"Q1: Is the red ball red? A: {query1.value.item():.4f}")

    # Query 2: Is the blue cube round?
    query2 = is_round(blue_cube)
    print(f"Q2: Is the blue cube round? A: {query2.value.item():.4f}")

    # Query 3: Are all balls round? (Universal quantification)
    all_balls_round = Forall(balls, is_round(balls))
    print(f"Q3: Are all balls round? A: {all_balls_round.value.item():.4f}")

In [222]:
# 8. EXECUTE O EXEMPLO COMPLETO
print("=== EXEMPLO DE RACIOCÍNIO LÓGICO LTN ===")
initial_sat = check_satisfiability()
train_predicates(epochs=100)
final_sat = check_satisfiability()
query_knowledge_base()
print("\n=== PRINCIPAIS INFORMAÇÕES SOBRE O LTNObjects ===")
print("1. LTNObjects encapsulam tensores com significado lógico.")
print("2. Constantes representam entidades específicas")
print("3. As variáveis representam conjuntos quantificáveis")
print("4. Os predicados mapeiam LTNObjects para valores de verdade [0,1]")
print("5. Os conectivos lógicos (E, Ou, Não, Implica) operam sobre valores de verdade.")
print("6. Os quantificadores (Forall, Exists) agregam sobre a variávels")
print("7. No LTNTorch, você deve definir explicitamente todos os conectivos lógicos.")
print("8. A função SatAgg combina múltiplos axiomas em uma única pontuação de satisfação.")


=== EXEMPLO DE RACIOCÍNIO LÓGICO LTN ===
Knowledge base satisfaction level: 0.5438

=== TRAINING PREDICATES ===
Epoch 0: Satisfaction = 0.5398
Epoch 20: Satisfaction = 0.6553
Epoch 40: Satisfaction = 0.7397
Epoch 60: Satisfaction = 0.7936
Epoch 80: Satisfaction = 0.8234
Knowledge base satisfaction level: 0.8421

=== QUERYING KNOWLEDGE BASE ===
Q1: Is the red ball red? A: 0.9633
Q2: Is the blue cube round? A: 0.3425
Q3: Are all balls round? A: 0.9216

=== PRINCIPAIS INFORMAÇÕES SOBRE O LTNObjects ===
1. LTNObjects encapsulam tensores com significado lógico.
2. Constantes representam entidades específicas
3. As variáveis representam conjuntos quantificáveis
4. Os predicados mapeiam LTNObjects para valores de verdade [0,1]
5. Os conectivos lógicos (E, Ou, Não, Implica) operam sobre valores de verdade.
6. Os quantificadores (Forall, Exists) agregam sobre a variávels
7. No LTNTorch, você deve definir explicitamente todos os conectivos lógicos.
8. A função SatAgg combina múltiplos axiomas em

In [223]:
# ==============================================================================
# EXTENSÃO: DIFERENTES FORMAS DE IGUALDADE EM LTN (Extensão Original)
# ==============================================================================
# In[45]:
print("\n" + "="*50)
print("EXTENSION: DIFFERENT FORMS OF EQUALITY IN LTN")
print("="*50)


EXTENSION: DIFFERENT FORMS OF EQUALITY IN LTN


In [224]:
Not_eq = ltn.Connective(ltn.fuzzy_ops.NotStandard())
And_eq = ltn.Connective(ltn.fuzzy_ops.AndProd())
Or_eq = ltn.Connective(ltn.fuzzy_ops.OrProbSum())
Implies_eq = ltn.Connective(ltn.fuzzy_ops.ImpliesReichenbach())
Forall_eq = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
Exists_eq = ltn.Quantifier(ltn.fuzzy_ops.AggregPMean(p=2), quantifier="e")
sat_agg_eq = ltn.fuzzy_ops.SatAgg()

In [225]:
red_object1 = ltn.Constant(torch.tensor([1.0, 0.0, 0.0], device=ltn.device), trainable=True)
red_object2 = ltn.Constant(torch.tensor([0.95, 0.05, 0.0], device=ltn.device), trainable=True)
blue_object = ltn.Constant(torch.tensor([0.0, 0.0, 1.0], device=ltn.device), trainable=True)

objects = ltn.Variable("objetos", torch.tensor([
    [1.0, 0.0, 0.0],
    [0.95, 0.05, 0.0],
    [0.0, 0.0, 1.0],
    [0.0, 1.0, 0.0],
], device=ltn.device))
print("Experimento sobre igualdade: configuração concluída!")

# In[46]: 1. Igualdade diagonal LTN
print("\n" + "-"*30)
print("1. IGUALDADE DIAGONAL LTN")
print("-"*30)

Experimento sobre igualdade: configuração concluída!

------------------------------
1. IGUALDADE DIAGONAL LTN
------------------------------


In [226]:
class DiagonalEquality(nn.Module):
    def forward(self, x, y):
        x_val = x.value if hasattr(x, 'value') else x
        y_val = y.value if hasattr(y, 'value') else y
        cos_sim = torch.nn.functional.cosine_similarity(x_val, y_val, dim=-1)
        return 0.5 * (cos_sim + 1.0)

In [227]:
# Teste da Igualdade diagonal euclidiana
Equal_Diag = ltn.Predicate(DiagonalEquality().to(ltn.device))

In [228]:
def axioms_withEquality_Diag():
    axioms = [
        Forall_eq(objects, Equal_Diag(objects, objects)),
        Equal_Diag(red_object1, red_object2),
        Equal_Diag(red_object2, red_object1),
        Implies_eq(And_eq(Equal_Diag(red_object1, red_object2), Equal_Diag(red_object2, red_object1)), Equal_Diag(red_object1, red_object1)),
        Not_eq(Equal_Diag(red_object1, blue_object))
    ]
    return sat_agg_eq(*axioms)

In [229]:
print("Testando a igualdade diagonal...")
sat_diag = axioms_withEquality_Diag()
print(f"Satisfação KB de igualdade diagonal: {sat_diag.item():.4f}")
print(f"P: Os dois objetos vermelhos são iguais? R: {Equal_Diag(red_object1, red_object2).value.item():.4f}")
print(f"P: Um objeto vermelho é igual a um objeto azul? R: {Equal_Diag(red_object1, blue_object).value.item():.4f}")

# In[50]: 2. Igualdade de cosseno
print("\n" + "-"*30)
print("2. IGUALDADE DE COSSENO.")
print("-"*30)

Testando a igualdade diagonal...
Satisfação KB de igualdade diagonal: 0.7764
P: Os dois objetos vermelhos são iguais? R: 0.9993
P: Um objeto vermelho é igual a um objeto azul? R: 0.5000

------------------------------
2. IGUALDADE DE COSSENO.
------------------------------


In [230]:
class CosineEquality(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.tensor(1.0, device=ltn.device))
    def forward(self, x, y):
        x_val = x.value if hasattr(x, 'value') else x
        y_val = y.value if hasattr(y, 'value') else y
        cos_sim = torch.nn.functional.cosine_similarity(x_val, y_val, dim=-1)
        return torch.sigmoid(self.temperature * (cos_sim + 1.0) / 2.0)

Equal_Cos = ltn.Predicate(CosineEquality().to(ltn.device))

In [231]:
def axioms_withEquality_Cos():
    parameters = list(Equal_Cos.model.parameters())
    optimizer = torch.optim.Adam(parameters, lr=0.1)
    print("Treinamento da Igualdade do Cosseno...")
    for epoch in range(200):
        optimizer.zero_grad()
        axioms = [Forall_eq(objects, Equal_Cos(objects, objects)), Equal_Cos(red_object1, red_object2), Equal_Cos(red_object2, red_object1), Not_eq(Equal_Cos(red_object1, blue_object))]
        sat = sat_agg_eq(*axioms)
        loss = 1 - sat
        loss.backward()
        optimizer.step()
        if epoch % 50 == 0:
            print(f"Época {epoch}: Satisfatibilidade da igualdade do cosseno = {sat.item():.4f}, Temp = {Equal_Cos.model.temperature.item():.4f}")
    return sat_agg_eq(Forall_eq(objects, Equal_Cos(objects, objects)), Equal_Cos(red_object1, red_object2), Not_eq(Equal_Cos(red_object1, blue_object)))

sat_cos = axioms_withEquality_Cos()
print(f"Satisfatibilidade KB da Igualdade do Cosseno Final: {sat_cos.item():.4f}")
print(f"Q: Os dois objetos vermelhos são iguais (cosseno)? A: {Equal_Cos(red_object1, red_object2).value.item():.4f}")
print(f"Q: O objeto vermelho é igual ao objeto azul (cosseno)? A: {Equal_Cos(red_object1, blue_object).value.item():.4f}")


Treinamento da Igualdade do Cosseno...
Época 0: Satisfatibilidade da igualdade do cosseno = 0.6112, Temp = 1.1000
Época 50: Satisfatibilidade da igualdade do cosseno = 0.6255, Temp = 1.5738
Época 100: Satisfatibilidade da igualdade do cosseno = 0.6255, Temp = 1.5680
Época 150: Satisfatibilidade da igualdade do cosseno = 0.6255, Temp = 1.5687
Satisfatibilidade KB da Igualdade do Cosseno Final: 0.5793
Q: Os dois objetos vermelhos são iguais (cosseno)? A: 0.8275
Q: O objeto vermelho é igual ao objeto azul (cosseno)? A: 0.6866


In [232]:
# In[51]: 3. Igualdade Euclidiana
print("\n" + "-"*30)
print("3. IGUALDADE EUCLIDIANA")
print("-"*30)


------------------------------
3. IGUALDADE EUCLIDIANA
------------------------------


In [233]:
class EuclideanEquality(nn.Module):
    def __init__(self):
        super().__init__()
        self.gamma = nn.Parameter(torch.tensor(0.5, device=ltn.device))
    def forward(self, x, y):
        x_val = x.value if hasattr(x, 'value') else x
        y_val = y.value if hasattr(y, 'value') else y
        distance = torch.sum(torch.square(x_val - y_val), dim=-1)
        return torch.exp(-self.gamma * distance)

Equal_Eucl = ltn.Predicate(EuclideanEquality().to(ltn.device))


In [234]:
def axioms_withEquality_Eucl():
    parameters = list(Equal_Eucl.model.parameters())
    optimizer = torch.optim.Adam(parameters, lr=0.1)
    print("Treinamento em Igualdade Euclidiana...")
    for epoch in range(150):
        optimizer.zero_grad()
        axioms = [Forall_eq(objects, Equal_Eucl(objects, objects)), Equal_Eucl(red_object1, red_object2), Equal_Eucl(red_object2, red_object1), Not_eq(Equal_Eucl(red_object1, blue_object))]
        sat = sat_agg_eq(*axioms)
        loss = 1 - sat
        loss.backward()
        optimizer.step()
        if epoch % 30 == 0:
            print(f"Época {epoch}: Satisfatibilidade da Igualdade Euclidiana = {sat.item():.4f}, Gamma = {Equal_Eucl.model.gamma.item():.4f}")
    return sat_agg_eq(Forall_eq(objects, Equal_Eucl(objects, objects)), Equal_Eucl(red_object1, red_object2), Not_eq(Equal_Eucl(red_object1, blue_object)))

sat_eucl = axioms_withEquality_Eucl()
print(f"Satisfatibilidade final da igualdade euclidiana KB: {sat_eucl.item():.4f}")
print(f"Q: Os dois objetos vermelhos são iguais (euclidianos)? A: {Equal_Eucl(red_object1, red_object2).value.item():.4f}")
print(f"Q: O objeto vermelho é igual ao objeto azul (euclidiano)? A: {Equal_Eucl(red_object1, blue_object).value.item():.4f}")


Treinamento em Igualdade Euclidiana...
Época 0: Satisfatibilidade da Igualdade Euclidiana = 0.8160, Gamma = 0.6000
Época 30: Satisfatibilidade da Igualdade Euclidiana = 0.9903, Gamma = 2.2782
Época 60: Satisfatibilidade da Igualdade Euclidiana = 0.9905, Gamma = 2.4738
Época 90: Satisfatibilidade da Igualdade Euclidiana = 0.9905, Gamma = 2.4658
Época 120: Satisfatibilidade da Igualdade Euclidiana = 0.9905, Gamma = 2.4479
Satisfatibilidade final da igualdade euclidiana KB: 0.9917
Q: Os dois objetos vermelhos são iguais (euclidianos)? A: 0.9879
Q: O objeto vermelho é igual ao objeto azul (euclidiano)? A: 0.0076


In [235]:
# In[52]: 4. Igualdade de Variedades Aprendíveis
print("\n" + "-"*30)
print("4. IGUALDADE DE VARIEDADES APRENDÍVEIS")
print("-"*30)

class LearnableEquality(nn.Module):
    def __init__(self, input_dim=3):
        super(LearnableEquality, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim * 2, 64), nn.ELU(),
            nn.Linear(64, 32), nn.ELU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x, y):
        x_val = x.value if hasattr(x, 'valor') else x
        y_val = y.value if hasattr(y, 'valor') else y
        cat_inputs = torch.cat([x_val, y_val], dim=-1)
        return self.net(cat_inputs).squeeze()

Equal_Learned = ltn.Predicate(LearnableEquality(input_dim=3).to(ltn.device))



------------------------------
4. IGUALDADE DE VARIEDADES APRENDÍVEIS
------------------------------


In [236]:
def axioms_withEquality_Learned():
    parameters = list(Equal_Learned.model.parameters())
    optimizer = torch.optim.Adam(parameters, lr=0.01)
    print("Treinaento da Rede de Igualdade aprodível...")
    for epoch in range(300):
        optimizer.zero_grad()
        axioms = [
            Forall_eq(objects, Equal_Learned(objects, objects)),
            Equal_Learned(red_object1, red_object2), Equal_Learned(red_object2, red_object1),
            Not_eq(Equal_Learned(red_object1, blue_object)), Not_eq(Equal_Learned(blue_object, red_object1)),
            Implies_eq(And_eq(Equal_Learned(red_object1, red_object2), Equal_Learned(red_object2, red_object1)), Equal_Learned(red_object1, red_object1))
        ]
        sat = sat_agg_eq(*axioms)
        loss = 1 - sat
        loss.backward()
        optimizer.step()
        if epoch % 50 == 0:
            print(f"Epoch {epoch}: Satisfatibilidade com a igualdade aprendível= {sat.item():.4f}")
    return sat_agg_eq(Forall_eq(objects, Equal_Learned(objects, objects)), Equal_Learned(red_object1, red_object2), Not_eq(Equal_Learned(red_object1, blue_object)))

sat_learned = axioms_withEquality_Learned()
print(f"Satisfação final com a base de conhecimento sobre igualdade aprendível: {sat_learned.item():.4f}")
print(f"Q:Os dois objetos vermelhos são iguais (Aprendido)? A: {Equal_Learned(red_object1, red_object2).value.item():.4f}")
print(f"Q: O objeto vermelho é igual ao objeto azul (aprendido)? A: {Equal_Learned(red_object1, blue_object).value.item():.4f}")
print(f"Q: O objeto vermelho é igual a si mesmo (Aprendido)? A: {Equal_Learned(red_object1, red_object1).value.item():.4f}")


Treinaento da Rede de Igualdade aprodível...
Epoch 0: Satisfatibilidade com a igualdade aprendível= 0.5140
Epoch 50: Satisfatibilidade com a igualdade aprendível= 0.7959
Epoch 100: Satisfatibilidade com a igualdade aprendível= 0.7959
Epoch 150: Satisfatibilidade com a igualdade aprendível= 0.7959
Epoch 200: Satisfatibilidade com a igualdade aprendível= 0.7959
Epoch 250: Satisfatibilidade com a igualdade aprendível= 0.7959
Satisfação final com a base de conhecimento sobre igualdade aprendível: 0.7113
Q:Os dois objetos vermelhos são iguais (Aprendido)? A: 1.0000
Q: O objeto vermelho é igual ao objeto azul (aprendido)? A: 0.0002
Q: O objeto vermelho é igual a si mesmo (Aprendido)? A: 1.0000


In [237]:
# ==============================================================================
# NOVA EXTENSÃO: RELAÇÕES GEOMÉTRICAS EM UM PLANO 2D (Para os Alunos)
# ==============================================================================
# In[53]:
print("\n" + "="*60)
print("EXTENSÃO: RELAÇÕES GEOMÉTRICAS EM UM PLANO 2D")
print("="*60)

# ==============================================================================
# DESAFIO PARA OS ALUNOS:
# Nesta extensão, você é convidado a formalizar relações geométricas
# usando Lógica de Primeira Ordem (FOL) e implementá-las no LTN.
#
# Definimos três classes de figuras geométricas 2D baseadas no centro (x,y):
# - Circle(x): A figura x é um círculo
# - Square(x): A figura x é um quadrado
# - Triangle(x): A figura x é um triângulo
#
# E relações espaciais entre pares de figuras:
# - LeftOf(x,y): A figura x está à esquerda da figura y
# - Above(x,y): A figura x está acima da figura y
# - Overlaps(x,y): A figura x se sobrepõe à figura y
#
# AXIOMAS EM LÓGICA DE PRIMEIRA ORDEM (Base de Conhecimento):
# 1. Todos os círculos estão à esquerda de todos os quadrados:
#    ∀x ∀y (Circle(x) ∧ Square(y) → LeftOf(x,y))
#
# 2. Todos os triângulos estão acima de todos os círculos:
#    ∀x ∀y (Triangle(x) ∧ Circle(y) → Above(x,y))
#
# 3. A sobreposição é simétrica:
#    ∀x ∀y (Overlaps(x,y) → Overlaps(y,x))
#
# 4. Nenhum objeto se sobrepõe a si mesmo:
#    ∀x ¬Overlaps(x,x)
#
# TAREFA DO ALUNO: Estenda esta base de conhecimento com novos axiomas! Ex:
# - ∀x ∀y (Square(x) ∧ Triangle(y) → ¬Overlaps(x,y))
# - ∀x (Circle(x) → ∃y (Square(y) ∧ LeftOf(x,y)))
# ==============================================================================

# 1. GERAÇÃO DE DADOS (Plano 2D Aleatório)
def generate_random_shapes(num_per_shape=10):
    circles = (
        torch.rand(num_per_shape, 2, device=ltn.device)
        * torch.tensor([4.0, 4.0], device=ltn.device)
    )

    squares = (
        torch.rand(num_per_shape, 2, device=ltn.device)
        * torch.tensor([4.0, 4.0], device=ltn.device)
        + torch.tensor([6.0, 0.0], device=ltn.device)
    )

    triangles = (
        torch.rand(num_per_shape, 2, device=ltn.device)
        * torch.tensor([10.0, 4.0], device=ltn.device)
        + torch.tensor([0.0, 6.0], device=ltn.device)
    )

    return circles, squares, triangles

# 2. PREDICADOS NEURAIS
class ShapeClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    def forward(self, x):
        x_val = x.value if hasattr(x, 'value') else x
        return self.net(x_val).squeeze()

class LearnableSpatialRelation(nn.Module):
    def __init__(self, feature_extractor):
        super().__init__()
        self.feature_extractor = feature_extractor
        self.net = nn.Sequential(
            nn.Linear(1, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    def forward(self, x, y):
      x_val = x.value if hasattr(x, 'value') else x
      y_val = y.value if hasattr(y, 'value') else y
      features = self.feature_extractor(x_val, y_val)
      return self.net(features).squeeze(-1)

def left_of_features(x, y):
    return (y[..., 0] - x[..., 0]).unsqueeze(-1)

def above_features(x, y):
    return (x[..., 1] - y[..., 1]).unsqueeze(-1)

def distance_features(x, y):
    return torch.norm(x - y, dim=-1, keepdim=True)

is_circle = ltn.Predicate(ShapeClassifier().to(ltn.device))
is_square = ltn.Predicate(ShapeClassifier().to(ltn.device))
is_triangle = ltn.Predicate(ShapeClassifier().to(ltn.device))
left_of = ltn.Predicate(LearnableSpatialRelation(left_of_features).to(ltn.device))
above = ltn.Predicate(LearnableSpatialRelation(above_features).to(ltn.device))
overlaps = ltn.Predicate(LearnableSpatialRelation(distance_features).to(ltn.device))

# 3. AXIOMAS LÓGICOS
def define_geo_axioms(circles, squares, triangles):
    c = ltn.Variable("c", circles)
    s = ltn.Variable("s", squares)
    t = ltn.Variable("t", triangles)
    axioms = [
        Forall([c, s], Implies(And(is_circle(c), is_square(s)), left_of(c, s))),
        Forall([t, c], Implies(And(is_triangle(t), is_circle(c)), above(t, c))),
        Forall([c, s], Implies(overlaps(c, s), overlaps(s, c))),
        Forall(c, Not(overlaps(c, c))),
        Forall(s, Not(overlaps(s, s))),
        Forall(t, Not(overlaps(t, t))),
        # Fatos positivos de classificação
        Forall(c, is_circle(c)),
        Forall(s, is_square(s)),
        Forall(t, is_triangle(t)),
        # Fatos negativos de classificação
        Forall(c, Not(is_square(c))),
        Forall(c, Not(is_triangle(c))),
        Forall(s, Not(is_circle(s))),
        Forall(s, Not(is_triangle(s))),
        Forall(t, Not(is_circle(t))),
        Forall(t, Not(is_square(t))),
        # Axioma extra: quadrados não se sobrepõem a triângulos
        Forall([s, t], Implies(
            And(is_square(s), is_triangle(t)),
            Not(overlaps(s, t))
        )),
        # Assimetria da relação LeftOf
        Forall([c, s], Implies(
            left_of(c, s),
            Not(left_of(s, c))
        )),

        # Assimetria da relação Above
        Forall([t, c], Implies(
            above(t, c),
            Not(above(c, t))
        )),
    ]
    return sat_agg(*axioms)

# 4. TREINAMENTO
def train_geometrical_model(epochs=200):
    parameters = []
    for pred in [is_circle, is_square, is_triangle, left_of, above, overlaps]:
        parameters += list(pred.model.parameters())
    optimizer = torch.optim.Adam(parameters, lr=0.01)
    circles, squares, triangles = generate_random_shapes(num_per_shape=10)
    print("\n=== TREINANDO AXIOMAS GEOMÉTRICOS ===")
    for epoch in range(epochs):
        optimizer.zero_grad()
        sat = define_geo_axioms(circles, squares, triangles)
        loss = 1 - sat
        loss.backward()
        optimizer.step()
        if epoch % 40 == 0:
            print(f"Epoch {epoch}: Satisfação = {sat.item():.4f}")
    return circles, squares, triangles

# 5. TESTE EM NOVOS DADOS ALEATÓRIOS
def test_geometrical_model():
    print("\n=== TESTANDO EM NOVOS DADOS ALEATÓRIOS ===")

    test_circles, test_squares, test_triangles = (
        generate_random_shapes(num_per_shape=20)
    )

    c = ltn.Variable("c_test", test_circles)
    s = ltn.Variable("s_test", test_squares)
    t = ltn.Variable("t_test", test_triangles)

    with torch.no_grad():

        queries = [
            ("Reconhecimento dos círculos",
             Forall(c, is_circle(c))),

            ("Reconhecimento dos quadrados",
             Forall(s, is_square(s))),

            ("Reconhecimento dos triângulos",
             Forall(t, is_triangle(t))),

            ("Círculos à esquerda dos quadrados",
             Forall([c, s], left_of(c, s))),

            ("Quadrados à esquerda dos círculos (falso)",
             Forall([s, c], left_of(s, c))),

            ("Triângulos acima dos círculos",
             Forall([t, c], above(t, c))),

            ("Círculos acima dos triângulos (falso)",
             Forall([c, t], above(c, t))),

            ("Quadrados não sobrepõem triângulos",
             Forall([s, t], Not(overlaps(s, t)))),

            ("Satisfação global no teste",
             define_geo_axioms(
                 test_circles,
                 test_squares,
                 test_triangles
             ))
        ]

        for i, (description, result) in enumerate(queries, 1):
          value = result.value if hasattr(result, "value") else result
          print(f"Q{i}: {description}: {value.item():.4f}")

train_geometrical_model(epochs=500)
test_geometrical_model()


EXTENSÃO: RELAÇÕES GEOMÉTRICAS EM UM PLANO 2D

=== TREINANDO AXIOMAS GEOMÉTRICOS ===
Epoch 0: Satisfação = 0.5459
Epoch 40: Satisfação = 0.8157
Epoch 80: Satisfação = 0.9204
Epoch 120: Satisfação = 0.9664
Epoch 160: Satisfação = 0.9848
Epoch 200: Satisfação = 0.9913
Epoch 240: Satisfação = 0.9944
Epoch 280: Satisfação = 0.9961
Epoch 320: Satisfação = 0.9971
Epoch 360: Satisfação = 0.9978
Epoch 400: Satisfação = 0.9982
Epoch 440: Satisfação = 0.9985
Epoch 480: Satisfação = 0.9988

=== TESTANDO EM NOVOS DADOS ALEATÓRIOS ===
Q1: Reconhecimento dos círculos: 0.9916
Q2: Reconhecimento dos quadrados: 0.9989
Q3: Reconhecimento dos triângulos: 0.9984
Q4: Círculos à esquerda dos quadrados: 0.9998
Q5: Quadrados à esquerda dos círculos (falso): 0.0000
Q6: Triângulos acima dos círculos: 0.9999
Q7: Círculos acima dos triângulos (falso): 0.0000
Q8: Quadrados não sobrepõem triângulos: 0.9999
Q9: Satisfação global no teste: 0.9265


## Análise dos resultados e conclusão

A extensão geométrica foi aprimorada por meio das seguintes contribuições:

- Correção da dimensionalidade dos tensores nos predicados espaciais.
- Inclusão de nove axiomas de classificação positiva e negativa das figuras.
- Reestruturação da distribuição espacial dos dados.
- Implementação do axioma de não sobreposição entre quadrados e triângulos.
- Inclusão de axiomas de assimetria para as relações `LeftOf` e `Above`.
- Ampliação do treinamento para 500 épocas.
- Expansão das consultas de avaliação em dados inéditos.

O modelo atingiu um grau de satisfação de 0,9988 na época 480 do treinamento e de 0,9265 no conjunto de teste.

As consultas espaciais positivas apresentaram valores superiores a 0,999, enquanto as relações inversas apresentaram valores inferiores a 0,001, demonstrando coerência com as restrições lógicas estabelecidas.

Os resultados fornecem evidências de generalização para novos objetos provenientes da mesma distribuição utilizada no treinamento.

Como limitações, destaca-se que a classificação das figuras depende de suas coordenadas espaciais, e não de suas características geométricas reais. Além disso, a ausência de exemplos positivos de sobreposição impede uma avaliação completa do predicado `Overlaps`.

Conclui-se que a integração entre redes neurais e lógica de primeira ordem permite incorporar restrições simbólicas ao processo de aprendizado, favorecendo a coerência lógica das inferências realizadas pelo modelo.